In [1]:
import torch 
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


/home/ashant/Desktop/75dayplan/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
document = """
Retrieval Augmented Generation (RAG) is a technique that combines retrieval systems with large language models.
RAG improves factual accuracy by grounding responses in external documents.
Vector databases like FAISS are used to store embeddings.
Cosine similarity measures the angle between vectors.
L2 distance measures Euclidean distance between vectors.
Chunking text helps preserve semantic meaning.
Overlap prevents context loss across chunks.
Hallucination occurs when the model generates unsupported information.
Precision measures correctness of retrieved documents.
Recall measures completeness of retrieval.
"""


In [4]:
def chunk_text(text,chunk_size=150,overlap=30):
    chunks =[]
    start=0
    while start <len(text):
        end =start + chunk_size
        chunks.append(text[start:end])
        start+=chunk_size-overlap #overlap is used to prevent semantic boundary loss

    return chunks

chunks = chunk_text(document)  
for i , chunk in enumerate(chunks):
    print(f"\nChunk{i}: \n{chunk}")


Chunk0: 

Retrieval Augmented Generation (RAG) is a technique that combines retrieval systems with large language models.
RAG improves factual accuracy by grou

Chunk1: 
roves factual accuracy by grounding responses in external documents.
Vector databases like FAISS are used to store embeddings.
Cosine similarity measu

Chunk2: 
dings.
Cosine similarity measures the angle between vectors.
L2 distance measures Euclidean distance between vectors.
Chunking text helps preserve sem

Chunk3: 
unking text helps preserve semantic meaning.
Overlap prevents context loss across chunks.
Hallucination occurs when the model generates unsupported in

Chunk4: 
model generates unsupported information.
Precision measures correctness of retrieved documents.
Recall measures completeness of retrieval.


Chunk5: 
ness of retrieval.



In [5]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

sample_embedding = embedder.encode(["test"])
print("Embedding dimension:",sample_embedding.shape[1])

Embedding dimension: 384


In [6]:
chunk_embeddings = embedder.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=False)
print(chunk_embeddings.shape)

(6, 384)


In [12]:
dimension =chunk_embeddings.shape[1]

faiss.normalize_L2(chunk_embeddings)
# print("Embedding shape:", chunk_embeddings.shape)

index_cosine = faiss.IndexFlatIP(dimension)
# print("Index dimension:", index_cosine.d)
index_cosine.add(chunk_embeddings)

print("Total vectors indexed: ",index_cosine.ntotal)

Total vectors indexed:  6


In [15]:
def retrieve(query,index,chunks,k=3):
    query_embeddings=embedder.encode([query],convert_to_numpy=True)
    faiss.normalize_L2(query_embeddings)
    scores,indices =index.search(query_embeddings,k)
    results =[chunks[i] for i in indices[0]]
    return results, scores

query ="What is cosione Similarity?"
results,scores = retrieve(query,index_cosine,chunks)

for r in results:
    print("\nRetrieved:\n",r)


Retrieved:
 dings.
Cosine similarity measures the angle between vectors.
L2 distance measures Euclidean distance between vectors.
Chunking text helps preserve sem

Retrieved:
 roves factual accuracy by grounding responses in external documents.
Vector databases like FAISS are used to store embeddings.
Cosine similarity measu

Retrieved:
 ness of retrieval.



In [16]:
#Build L2 Index
chunk_embeddings_l2= embedder.encode(
    chunks,
    convert_to_numpy= True,
    normalize_embeddings= False)
index_l2 = faiss.IndexFlatL2(dimension)
index_l2.add(chunk_embeddings_l2)

query_embedding = embedder.encode([query],convert_to_numpy=True)
scores_l2,indices_l2 = index_l2.search(query_embedding,3)

print("L2 Results:")
for i in indices_l2[0]:
    print("\n",chunks[i])

L2 Results:

 dings.
Cosine similarity measures the angle between vectors.
L2 distance measures Euclidean distance between vectors.
Chunking text helps preserve sem

 roves factual accuracy by grounding responses in external documents.
Vector databases like FAISS are used to store embeddings.
Cosine similarity measu

 ness of retrieval.



In [19]:
relevant_ids ={3}
retrieved_ids=set(indices_l2[0])
def precision_at_k(retrieved,relevant):
    return len(retrieved & relevant)/len(retrieved)

def recall_at_k(retrieved , relevant):
    return len(retrieved & relevant)/len(relevant)

print("Precision: ",precision_at_k(retrieved_ids,relevant_ids))

print("Recall: ",recall_at_k(retrieved_ids,relevant_ids))

Precision:  0.0
Recall:  0.0


In [21]:
def generate_answer(query, retrieved_chunks):
    context = "\n".join(retrieved_chunks)
    
    prompt = f"""
    Answer ONLY using the context below.
    If answer not found, say 'Not in document.'

    Context:
    {context}

    Question:
    {query}
    """
    
    return prompt  # Replace with real LLM later

print(generate_answer(query,results))


    Answer ONLY using the context below.
    If answer not found, say 'Not in document.'

    Context:
    dings.
Cosine similarity measures the angle between vectors.
L2 distance measures Euclidean distance between vectors.
Chunking text helps preserve sem
roves factual accuracy by grounding responses in external documents.
Vector databases like FAISS are used to store embeddings.
Cosine similarity measu
ness of retrieval.


    Question:
    What is cosione Similarity?
    
